# Notebook 4 — Frame Extraction & DinoTXT Vision Embedding Pre-computation

For each annotated segment, extracts frames from the video, runs each frame through
the frozen DinoTXT vision encoder (`encode_image`), and saves the mean-pooled
video embedding to disk.

**Design rationale (from DINOv2 paper section 7.2 and GitHub issue #16):**
- The DINOv2 authors evaluated video action recognition by sampling evenly spaced
  frames and training a linear classifier on the **average** of the frame features,
  without any fine-tuning of the backbone
- The paper author (pierrefdz) confirmed: N=8 frames, single linear classifier,
  no fine-tuning. Reference implementation runs backbone under torch.no_grad()
  and averages over the time dimension before the classifier
- We use up to 50 frames (more than 8) because YouCook2 segments are longer
  and more fine-grained than UCF/K-400. More frames = better coverage of the action

**What we save per segment:**
- `video_embedding`: shape (SHARED_DIM,) — mean of encode_image outputs across all frames

This is the final output of the full frozen DinoTXT vision tower (backbone + VisionHead).
It lives in the shared embedding space alongside the text embeddings computed in NB5.

**Model:** `dinov2_vitl14_reg4_dinotxt_tet1280d20h24l` — pretrained DinoTXT

**Reads:** `processed/segments_labeled.csv`, `annotations/segments_available.csv`
**Outputs:** `embeddings/dinov2/{segment_id}.pt` — tensor of shape (SHARED_DIM,)


In [2]:
import os, sys, json
import torch
import pandas as pd

with open('/home/ubuntu/youcook2-videos/paths.json') as f:
    PATHS = json.load(f)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

DINOV2_REPO = os.path.join(PATHS['repos'], 'dinov2')
if DINOV2_REPO not in sys.path:
    sys.path.insert(0, DINOV2_REPO)
import dinov2
print(f"dinov2 repo on sys.path ✓")


Device : cuda
GPU    : NVIDIA A10
dinov2 repo on sys.path ✓


## Load data — filter to available videos


In [4]:
df = pd.read_csv('/home/ubuntu/youcook2-videos/annotations/segments_verbs.csv')

print(f"Total segments: {len(df):,}")
print(df.head())


Total segments: 13,029
                                  caption     video_id       segment_id  \
0               crush and chop the garlic  --bv0V6ZjWI  --bv0V6ZjWI_000   
1       add oil garlic and salt to a bowl  --bv0V6ZjWI  --bv0V6ZjWI_001   
2  mix the tomoatoes with the oil mixture  --bv0V6ZjWI  --bv0V6ZjWI_002   
3                          chop the basil  --bv0V6ZjWI  --bv0V6ZjWI_003   
4   spread the oil mixture onto the dough  --bv0V6ZjWI  --bv0V6ZjWI_004   

  extracted_verb                             masked_caption  
0          crush                 [MASK] and chop the garlic  
1            add       [MASK] oil garlic and salt to a bowl  
2            mix  [MASK] the tomoatoes with the oil mixture  
3           chop                           [MASK] the basil  
4         spread      [MASK] the oil mixture onto the dough  


In [5]:
df_verbs = pd.read_csv('/home/ubuntu/youcook2-videos/annotations/segments_verbs.csv')
df_segs  = pd.read_csv('/home/ubuntu/youcook2-videos/annotations/segments.csv')

df = df_verbs.merge(df_segs[['segment_id', 'start_sec', 'end_sec']], on='segment_id', how='left')

print(f"Total segments: {len(df):,}")
print(f"Missing start/end: {df['start_sec'].isna().sum():,}")
print(df.head())

df.to_csv('/home/ubuntu/youcook2-videos/annotations/segments_verbs_timed.csv', index=False)
print("Saved → segments_verbs_timed.csv")

Total segments: 13,029
Missing start/end: 0
                                  caption     video_id       segment_id  \
0               crush and chop the garlic  --bv0V6ZjWI  --bv0V6ZjWI_000   
1       add oil garlic and salt to a bowl  --bv0V6ZjWI  --bv0V6ZjWI_001   
2  mix the tomoatoes with the oil mixture  --bv0V6ZjWI  --bv0V6ZjWI_002   
3                          chop the basil  --bv0V6ZjWI  --bv0V6ZjWI_003   
4   spread the oil mixture onto the dough  --bv0V6ZjWI  --bv0V6ZjWI_004   

  extracted_verb                             masked_caption  start_sec  \
0          crush                 [MASK] and chop the garlic        8.0   
1            add       [MASK] oil garlic and salt to a bowl       23.0   
2            mix  [MASK] the tomoatoes with the oil mixture       64.0   
3           chop                           [MASK] the basil       81.0   
4         spread      [MASK] the oil mixture onto the dough       90.0   

   end_sec  
0     23.0  
1     47.0  
2     76.0  
3     90

## Load pretrained DinoTXT — freeze everything

We load the full pretrained DinoTXT model and freeze all parameters.
Nothing in the model trains in NB4 or NB5 — this is purely evaluation
of the pretrained representations, as described in DINOv2 section 7.2.

We call `dinotxt.encode_image(frame)` per frame. This runs:
  frozen DINOv2 backbone → frozen VisionHead → normalised embedding
The output lives in the shared vision-language embedding space.

We read SHARED_DIM from the model config and save it to nb4_constants.json
so NB5 can load it without reloading the full model.


In [10]:
from dinov2.hub.dinotxt import dinov2_vitl14_reg4_dinotxt_tet1280d20h24l, get_tokenizer
from dinov2.data.transforms import make_classification_eval_transform

dinotxt = dinov2_vitl14_reg4_dinotxt_tet1280d20h24l().to(DEVICE)
dinotxt.eval()

for p in dinotxt.parameters():
    p.requires_grad = False

SHARED_DIM     = dinotxt.model_config.embed_dim
CONTEXT_LENGTH = 77

trainable = sum(p.numel() for p in dinotxt.parameters() if p.requires_grad)
total     = sum(p.numel() for p in dinotxt.parameters())
print(f"Model loaded: {total/1e6:.1f}M parameters, {trainable} trainable (must be 0)")
assert trainable == 0
print(f"SHARED_DIM : {SHARED_DIM}")
print(f"All parameters frozen ✓")

nb4_constants = {
    'MODEL_NAME':     'dinov2_vitl14_reg4_dinotxt_tet1280d20h24l',
    'SHARED_DIM':     SHARED_DIM,
    'CONTEXT_LENGTH': CONTEXT_LENGTH,
}
with open(os.path.join(PATHS['processed'], 'nb4_constants.json'), 'w') as f:
    json.dump(nb4_constants, f, indent=2)
print(json.dumps(nb4_constants, indent=2))

Could not import `tifffile`, TIFFImageDataDecoder will be disabled
Model loaded: 867.8M parameters, 0 trainable (must be 0)
SHARED_DIM : 2048
All parameters frozen ✓
{
  "MODEL_NAME": "dinov2_vitl14_reg4_dinotxt_tet1280d20h24l",
  "SHARED_DIM": 2048,
  "CONTEXT_LENGTH": 77
}


## Image preprocessing



In [11]:
from dinov2.data.transforms import make_classification_eval_transform

preprocess = make_classification_eval_transform()
print("Using official DinoTXT classification eval transform ✓")


Using official DinoTXT classification eval transform ✓


## Video embedding function

Mirrors the reference implementation from pierrefdz (GitHub issue #16):

```python
with torch.no_grad():
    output = model(frames)           # run backbone on all frames
    output = output.mean(dim=-2)     # average over time dimension
output = linear_classifier(output)   # classify
```

We use `dinotxt.encode_image` which runs the full frozen vision tower
(backbone + VisionHead) and returns a normalised embedding in the shared space.
Frames are processed in a batch for efficiency.


In [14]:
import numpy as np
from decord import VideoReader, cpu
from PIL import Image

MAX_FRAMES = 50  # max frames per segment
            # Author used 8 for UCF/K-400; we use 50 because YouCook2
            # segments are longer and cooking verbs are more fine-grained

# Read IMAGE_SIZE from the official transform 
IMAGE_SIZE = preprocess.transforms[1].size[0]  # CenterCrop size, not Resize
print(f"IMAGE_SIZE from official transform: {IMAGE_SIZE}")
# print all transforms so we can see the structure
for i, t in enumerate(preprocess.transforms):
    print(f"transforms[{i}]: {t}")

def extract_frames(video_path: str, start_sec: float, end_sec: float):
    """Extract up to MAX_FRAMES evenly spaced frames within the segment."""
    vr        = VideoReader(video_path, ctx=cpu(0))
    fps       = vr.get_avg_fps()
    start_idx = max(0, int(start_sec * fps))
    end_idx   = min(len(vr) - 1, int(end_sec * fps))
    if start_idx >= end_idx:
        return [Image.fromarray(vr[start_idx].asnumpy())]
    n_frames = min(MAX_FRAMES, end_idx - start_idx + 1)
    indices  = np.linspace(start_idx, end_idx, n_frames, dtype=int)
    frames   = vr.get_batch(indices).asnumpy()  # (n_frames, H, W, 3)
    return [Image.fromarray(frames[i]) for i in range(len(frames))]


@torch.no_grad()
def compute_video_embedding(video_path: str, start_sec: float, end_sec: float) -> torch.Tensor:
    """
    Runs the frozen DinoTXT vision tower on all frames and returns
    the mean-pooled embedding.

    Following pierrefdz's reference implementation:
      output = model(frames)       # shape: (n_frames, SHARED_DIM)
      output = output.mean(dim=0)  # shape: (SHARED_DIM,)

    encode_image handles: backbone → VisionHead → L2 normalisation
    """
    frames = extract_frames(video_path, start_sec, end_sec)

    # Stack all frames into a batch: (n_frames, 3, 224, 224)
    # frames are already PIL images — preprocess handles the rest
    frame_batch = torch.stack(
        [preprocess(frame) for frame in frames]
    ).to(DEVICE)

    # Run full frozen vision tower on all frames in one batch
    # encode_image output shape: (n_frames, SHARED_DIM), L2-normalised
    frame_embeddings = dinotxt.encode_image(frame_batch, normalize=True)

    # Mean pool over time dimension → (SHARED_DIM,)
    video_embedding = frame_embeddings.mean(dim=0)

    return video_embedding.cpu()


# Shape and value sanity check
dummy_frames = torch.randn(4, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
with torch.no_grad():
    test_emb = dinotxt.encode_image(dummy_frames, normalize=True)
print(f"encode_image output shape  : {test_emb.shape}  (expected: (4, {SHARED_DIM}))")
print(f"encode_image output L2 norm: {test_emb.norm(dim=-1).mean():.4f}  (expected: ~1.0)")
assert test_emb.shape == (4, SHARED_DIM)
assert abs(test_emb.norm(dim=-1).mean().item() - 1.0) < 0.01, "Output should be L2-normalised"
print("Shape and normalisation check ✓")

IMAGE_SIZE from official transform: 224
transforms[0]: Resize(size=256, interpolation=bicubic, max_size=None, antialias=True)
transforms[1]: CenterCrop(size=(224, 224))
transforms[2]: MaybeToTensor()
transforms[3]: Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
encode_image output shape  : torch.Size([4, 2048])  (expected: (4, 2048))
encode_image output L2 norm: 1.0000  (expected: ~1.0)
Shape and normalisation check ✓


## Compute and save video embeddings for all segments
Fully resumable — already computed files are skipped.


In [15]:
print(df.columns.tolist())
print(df.head(2))

['caption', 'video_id', 'segment_id', 'extracted_verb', 'masked_caption', 'start_sec', 'end_sec']
                             caption     video_id       segment_id  \
0          crush and chop the garlic  --bv0V6ZjWI  --bv0V6ZjWI_000   
1  add oil garlic and salt to a bowl  --bv0V6ZjWI  --bv0V6ZjWI_001   

  extracted_verb                        masked_caption  start_sec  end_sec  
0          crush            [MASK] and chop the garlic        8.0     23.0  
1            add  [MASK] oil garlic and salt to a bowl       23.0     47.0  


In [16]:
from tqdm.notebook import tqdm

EMB_DIR = PATHS['embeddings']
os.makedirs(EMB_DIR, exist_ok=True)

failed   = []
skipped  = 0
computed = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc="Computing video embeddings"):
    out_path = os.path.join(EMB_DIR, f"{row['segment_id']}.pt")
    if os.path.exists(out_path):
        skipped += 1
        continue
    vid_path = os.path.join(PATHS['videos'], f"{row['video_id']}.mp4")
    try:
        emb = compute_video_embedding(vid_path, row['start_sec'], row['end_sec'])
        torch.save(emb, out_path)
        computed += 1
    except Exception as e:
        failed.append({'segment_id': row['segment_id'], 'error': str(e)})

print(f"Computed : {computed:,}")
print(f"Skipped  : {skipped:,}  (already on disk)")
print(f"Failed   : {len(failed):,}")
if failed:
    for f in failed[:10]:
        print(f"  {f['segment_id']}: {f['error']}")


Computing video embeddings:   0%|          | 0/13029 [00:00<?, ?it/s]

[h264 @ 0x181f43040] mmco: unref short failure
[h264 @ 0x181f43040] mmco: unref short failure
[h264 @ 0x19e15b1c0] mmco: unref short failure
[h264 @ 0x19e15b1c0] mmco: unref short failure
[h264 @ 0x7d03d41dd140] mmco: unref short failure
[h264 @ 0x7d03d41dd140] mmco: unref short failure
[h264 @ 0x194d30480] mmco: unref short failure
[h264 @ 0x1a2af81c0] mmco: unref short failure
[h264 @ 0x19e15b1c0] mmco: unref short failure
[h264 @ 0x19e15b1c0] mmco: unref short failure
[h264 @ 0x1a2af81c0] mmco: unref short failure
[h264 @ 0x1a2af81c0] mmco: unref short failure
[h264 @ 0x7d03bc7603c0] mmco: unref short failure
[h264 @ 0x7d03bc7603c0] mmco: unref short failure
[h264 @ 0x7d03bc7603c0] mmco: unref short failure
[h264 @ 0x7d03bc7603c0] mmco: unref short failure
[h264 @ 0x1a2b0aa00] mmco: unref short failure
[h264 @ 0x1a2b0aa00] mmco: unref short failure
[h264 @ 0x1a2b0aa00] mmco: unref short failure
[h264 @ 0x181dc3b00] mmco: unref short failure
[h264 @ 0x181dc3b00] mmco: unref short fai

Computed : 13,003
Skipped  : 0  (already on disk)
Failed   : 26
  RubyHelAHBE_000: [01:05:54] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:104: Check failed: run_.load()
  RubyHelAHBE_001: [01:05:54] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:104: Check failed: run_.load()
  RubyHelAHBE_002: [01:05:54] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:104: Check failed: run_.load()
  RubyHelAHBE_003: [01:05:54] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:104: Check failed: run_.load()
  RubyHelAHBE_004: [01:05:54] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:104: Check failed: run_.load()
  RubyHelAHBE_005: [01:05:54] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:292: [01:05:54] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:218: Check failed: avcodec_send_packet(dec_ctx_.get(), pkt.get()) >= 0 (-11 vs. 0) Thread worker: Error sending packet.
  oC5OvA4BK-E_000: [03:47:58] /github/workspace/src/video/ffmpeg/threaded_decode

In [17]:
# Verify a sample saved embedding
sample_row  = df.sample(1).iloc[0]
sample_path = os.path.join(EMB_DIR, f"{sample_row['segment_id']}.pt")
emb         = torch.load(sample_path)

print(f"Segment  : {sample_row['segment_id']}")
print(f"Caption  : {sample_row['caption']}")
print(f"Shape    : {emb.shape}  (expected: torch.Size([{SHARED_DIM}]))")
print(f"L2 norm  : {emb.norm():.4f}  (expected: ~1.0, already normalised by encode_image)")
print(f"All zeros: {(emb == 0).all()}  (must be False)")

assert emb.shape == torch.Size([SHARED_DIM]), "Wrong embedding shape"
assert not (emb == 0).all(), "Embedding is all zeros"
print("\nEmbedding file looks correct ✓")


Segment  : 5cn9KJfaQXk_012
Caption  : cut the pizzas into 8 slices
Shape    : torch.Size([2048])  (expected: torch.Size([2048]))
L2 norm  : 0.9276  (expected: ~1.0, already normalised by encode_image)
All zeros: False  (must be False)

Embedding file looks correct ✓


In [19]:
df_verbs = pd.read_csv('/home/ubuntu/youcook2-videos/annotations/segments_verbs.csv')
df_segs  = pd.read_csv('/home/ubuntu/youcook2-videos/annotations/segments.csv')

df = df_verbs.merge(
    df_segs[['segment_id', 'start_sec', 'end_sec', 'subset']], 
    on='segment_id', 
    how='left'
)

print(f"Total segments: {len(df):,}")
print(f"Missing start/end: {df['start_sec'].isna().sum():,}")
print(f"Columns: {df.columns.tolist()}")
print(df.head())

df.to_csv('/home/ubuntu/youcook2-videos/annotations/segments_verbs_timed.csv', index=False)
print("Saved → segments_verbs_timed.csv")

Total segments: 13,029
Missing start/end: 0
Columns: ['caption', 'video_id', 'segment_id', 'extracted_verb', 'masked_caption', 'start_sec', 'end_sec', 'subset']
                                  caption     video_id       segment_id  \
0               crush and chop the garlic  --bv0V6ZjWI  --bv0V6ZjWI_000   
1       add oil garlic and salt to a bowl  --bv0V6ZjWI  --bv0V6ZjWI_001   
2  mix the tomoatoes with the oil mixture  --bv0V6ZjWI  --bv0V6ZjWI_002   
3                          chop the basil  --bv0V6ZjWI  --bv0V6ZjWI_003   
4   spread the oil mixture onto the dough  --bv0V6ZjWI  --bv0V6ZjWI_004   

  extracted_verb                             masked_caption  start_sec  \
0          crush                 [MASK] and chop the garlic        8.0   
1            add       [MASK] oil garlic and salt to a bowl       23.0   
2            mix  [MASK] the tomoatoes with the oil mixture       64.0   
3           chop                           [MASK] the basil       81.0   
4         spread  

In [20]:
# Identify failed segments and save a clean filtered dataframe
failed_ids    = {f['segment_id'] for f in failed}
failed_videos = {sid.rsplit('_', 1)[0] for sid in failed_ids}

print(f"Failed segment IDs  : {failed_ids}")
print(f"Failed video IDs    : {failed_videos}")

# Filter df to only segments with a computed embedding
df_clean = df[~df['segment_id'].isin(failed_ids)].reset_index(drop=True)

print(f"\nOriginal segments : {len(df):,}")
print(f"Failed            : {len(failed_ids):,}")
print(f"Clean segments    : {len(df_clean):,}")

# Save for use in NB5
clean_path = '/home/ubuntu/youcook2-videos/annotations/segments_verbs_timed_clean.csv'
df_clean.to_csv(clean_path, index=False)
print(f"Saved → segments_verbs_timed_clean.csv")

Failed segment IDs  : {'RubyHelAHBE_001', 'RubyHelAHBE_002', 'RubyHelAHBE_003', 'oC5OvA4BK-E_011', 'oC5OvA4BK-E_002', 'RubyHelAHBE_000', 'r4MPTWizETs_004', 'r4MPTWizETs_002', 'oC5OvA4BK-E_003', 'r4MPTWizETs_000', 'oC5OvA4BK-E_009', 'r4MPTWizETs_003', 'oC5OvA4BK-E_004', 'oC5OvA4BK-E_010', 'r4MPTWizETs_006', 'oC5OvA4BK-E_005', 'r4MPTWizETs_001', 'oC5OvA4BK-E_006', 'oC5OvA4BK-E_008', 'RubyHelAHBE_005', 'oC5OvA4BK-E_000', 'oC5OvA4BK-E_001', 'oC5OvA4BK-E_012', 'RubyHelAHBE_004', 'oC5OvA4BK-E_007', 'r4MPTWizETs_005'}
Failed video IDs    : {'r4MPTWizETs', 'oC5OvA4BK-E', 'RubyHelAHBE'}

Original segments : 13,029
Failed            : 26
Clean segments    : 13,003
Saved → segments_verbs_timed_clean.csv


## Done ✓
Proceed to **Notebook 5** for text embedding + classifier training.

**What was produced:**
- `embeddings/dinov2/{segment_id}.pt` — (SHARED_DIM,) float32 tensor per segment
  Mean of encode_image outputs across all frames. L2-normalised.
- `processed/nb4_constants.json` — SHARED_DIM, MODEL_NAME, CONTEXT_LENGTH

**Key points for NB5:**
- Everything in DinoTXT is frozen — nothing trains in NB5 except the classifier
- Text embeddings are computed on-the-fly in NB5 (fast, no need to pre-compute)
- Classifier input: cat([video_emb, text_emb]) shape (2 * SHARED_DIM,)
- Classifier output: logits shape (114,)
